# 01 — Data Pipeline

## Task 1.2 — Open-Meteo weather fetch

Fetches hourly `temperature_2m` and `shortwave_radiation` (GHI) from the
Open-Meteo historical archive for a given location and date range.

In [ ]:
import ssl
import requests
import pandas as pd
from requests.adapters import HTTPAdapter

In [ ]:
class _WinCertAdapter(HTTPAdapter):
    """
    Mounts Windows system CA store so requests works behind corporate proxies.
    """
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context()
        ctx.load_default_certs(ssl.Purpose.SERVER_AUTH)
        kwargs["ssl_context"] = ctx
        super().init_poolmanager(*args, **kwargs)

_session = requests.Session()
_session.mount("https://", _WinCertAdapter())


def fetch_weather(lat, lon, start_date, end_date, timezone) -> pd.DataFrame:
    """
    Returns hourly UTC-aware DataFrame indexed by timestamp,
       with columns: temperature_2m, shortwave_radiation (Global Horizontal Irradiance, W/m²).

    Data is fetched and stored in UTC regardless of the timezone arg.
    Convert to local time only for plotting — solar physics works in UTC + longitude.
    """
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,shortwave_radiation",
        "timezone": "UTC",  # always fetch UTC — avoids DST NonExistentTime/AmbiguousTime errors
    }
    resp = _session.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params,
        timeout=30,
    )
    resp.raise_for_status()

    hourly = resp.json()["hourly"]
    df = pd.DataFrame({
        "temperature_2m":      hourly["temperature_2m"],
        # shortwave_radiation == GHI (Global Horizontal Irradiance, W/m²)
        # pvlib will decompose this into DNI + DHI components
        "shortwave_radiation": hourly["shortwave_radiation"],
    }, index=pd.to_datetime(hourly["time"], utc=True))  # tz-aware UTC in one step, no tz_localize
    df.index.name = "timestamp"
    return df

In [ ]:
# Austin, TX — single source of truth; Week 2 pvlib uses these same coords
AUSTIN_LAT = 30.2672
AUSTIN_LON  = -97.7431
LOCAL_TZ    = "America/Chicago"

In [ ]:
# Smoke test — full year to exercise DST transitions
df_weather = fetch_weather(
    lat=AUSTIN_LAT,
    lon=AUSTIN_LON,
    start_date="2018-01-01",
    end_date="2018-12-31",
    timezone=LOCAL_TZ,
)

assert df_weather.index.tz is not None,               "Index must be tz-aware"
assert not df_weather.index.has_duplicates,            "Duplicate timestamps (DST fold?)"
assert df_weather.index.is_monotonic_increasing,       "Index not sorted"
assert df_weather.isna().sum().sum() == 0 or print("NaNs present — investigate")

print(f"Rows: {len(df_weather)} (expect ~8760 for a year)")
print(f"Index dtype: {df_weather.index.dtype}")
print(df_weather.head())